# 01. OpenAI API 기초
**SK하이닉스 Autonomous R&D — Day 3** 

> ChatGPT 웹과 **같은 모델**을 코드에서 호출.

---
## 0. 라이브러리 설치

아래 셀을 **최초 1회** 실행.

In [5]:
!pip install openai==1.58.1

In [6]:
!pip install openai python-dotenv

---
## 1. LLM API란?

강의자료 요약:

| 구분 | ChatGPT 웹 | API |
|------|------------|-----|
| 사용자 | 사람이 직접 입력 | **프로그램**이 요청 |
| 결과 | 화면에 표시 | **response 객체**로 반환 |
| 본질 | 대화 | **Prompt → Text** |

API 요청의 핵심 3요소:
1. **`model`** — 어떤 LLM을 쓸지
2. **`messages`** — system / user (대화 내용)
3. **`temperature`** — 답변의 무작위성 (0=일관, 1=창의)

---
## 2. API 키 연결


처음에는 **키를 변수에 넣어** 바로 연결해 봅니다.

> OpenAI Platform → [API keys](https://platform.openai.com/api-keys) 에서 발급

In [22]:
from openai import OpenAI

import os

# .env 파일에 OPENAI_API_KEY=sk-... 를 설정하세요
api_key = os.getenv('OPENAI_API_KEY')
if not api_key:
    raise ValueError('.env에 OPENAI_API_KEY=sk-... 를 설정하세요.')

client = OpenAI(api_key=api_key)

In [23]:
# 연결 테스트
test = client.chat.completions.create(
    model='gpt-4o-mini',
    messages=[{'role': 'user', 'content': 'Hi, reply with one word: OK'}],
    max_tokens=10,
)
print('연결 성공:', test.choices[0].message.content)

연결 성공: OK


1. **`.env` 파일**에 키 저장 (코드와 분리) (OPENAI_API_KEY=sk...)
2. **`.gitignore`**에 `.env` 등록 → Git에 **절대 올리지 않음**

In [19]:
from pathlib import Path

root = Path('..')  # ttest/ #상위폴더로 가도록 ../.. -> 상위폴더 두번
env_path = root / '.env'
gitignore_path = root / '.gitignore'

print('.env 존재:', env_path.exists(), '→', env_path.resolve())
print('.gitignore 존재:', gitignore_path.exists())

if gitignore_path.exists():
    content = gitignore_path.read_text(encoding='utf-8')
    print('.env in .gitignore:', '.env' in content)

.env 존재: True → C:\Users\Admin\OneDrive\바탕 화면\실습\.env
.gitignore 존재: True
.env in .gitignore: True


In [20]:
from dotenv import load_dotenv 
import os

load_dotenv(root / '.env') #.env 파일 가져오기 load_dotenv(경로)
api_key = os.getenv('OPENAI_API_KEY') #API KEY 가져오기

if not api_key:
    raise ValueError('ttest/.env에 OPENAI_API_KEY=sk-... 를 설정하세요.')

client = OpenAI(api_key=api_key) #연결고리 OpenAI 불러오기
print('✅ .env에서 API 키 로드 완료 (코드에 키 없음)')

✅ .env에서 API 키 로드 완료 (코드에 키 없음)


---
## 3. 첫 API 호출 

`chat.completions.create()` — **messages** 리스트를 보내면 AI가 답변합니다.

| role | 역할 |
|------|------|
| `system` | AI 역할·규칙 (선택) |
| `user` | 사용자 질문 |

In [ ]:
response = client.chat.completions.create(
    model='gpt-4o-mini',
    temperature=0.1,
    messages=[
        {'role': 'system', 'content': 'You are a helpful assistant.'},
        {'role': 'user',   'content': '2002년 월드컵 우승 팀이 어디야?'},
    ],
)
# system : 지시, 역할 등
# user : 실제 질문

In [28]:
response.choices[0].message.content

'2002년 월드컵 우승 팀은 브라질입니다. 브라질은 결승에서 독일을 2-0으로 이기고 5번째 월드컵 트로피를 차지했습니다.'

In [29]:
####### 2022년 월드컵 우승팀 물어보기
response = client.chat.completions.create(
    model = 'gpt-4o-mini',
    temperature = 0.1,
    messages=[
        {'role': 'system', 'content' : 'You are a helpful assistant'},
        {'role': 'user', 'content':'2022년 월드컵 우승 팀이 어디야?'}
    ],
)


...

Ellipsis

In [33]:
response.choices[0].message.content

'2022년 FIFA 월드컵 우승 팀은 아르헨티나입니다. 아르헨티나는 결승에서 프랑스를 상대로 승리하여 36년 만에 월드컵 트로피를 차지했습니다.'

---
## 4. 응답(Response) 객체

API는 **텍스트만**이 아니라 **객체**를 돌려줍니다.

| 필드 | 의미 |
|------|------|
| `response.choices[0].message.content` | **실제 답변 텍스트** ← 가장 많이 사용 |
| `response.choices[0].finish_reason` | `stop`=정상 종료, `length`=토큰 초과 |
| `response.usage.total_tokens` | 이번 호출에 쓴 토큰 수 (비용 참고) |
| `response.id` | 요청 ID (로그·디버깅용) |

In [34]:
print('ID:', response.id)
print('finish_reason:', response.choices[0].finish_reason)
print('답변:', response.choices[0].message.content)
print('토큰 사용:', response.usage.total_tokens, '(prompt:', response.usage.prompt_tokens,
      '/ completion:', response.usage.completion_tokens, ')')

ID: chatcmpl-DuBSxY1EH7cVLgtqM1lmKU7kITHuf
finish_reason: stop
답변: 2022년 FIFA 월드컵 우승 팀은 아르헨티나입니다. 아르헨티나는 결승에서 프랑스를 상대로 승리하여 36년 만에 월드컵 트로피를 차지했습니다.
토큰 사용: 81 (prompt: 29 / completion: 52 )


---
## 5. System / User 프롬프트 

같은 `user` 질문이라도 **`system`에 역할·출력 규칙**을 주면 답변 **형식·깊이·관점**이 달라집니다.

아래는 **의도적으로 짧고 모호한 질문**을 사용합니다. system 유무에 따라 차이가 잘 보입니다.

| | system | user |
|---|--------|------|
| 역할 | 모델의 **역할·규칙·형식** | 이번 **질문·작업** |
| 비유 | 직무 기술서 | 오늘 업무 지시 |

In [35]:
# 질문을 짧고 모호하게 → system이 없으면 범용 답변, 있으면 역할·형식에 맞춘 답변
question = 'for문이 뭐야?'

r1 = client.chat.completions.create(
    model='gpt-4o-mini',
    temperature=0.2,
    messages=[{'role': 'user', 'content': question}],
)

print('=== system 없음 (범용 설명, 길어질 수 있음) ===')
print(r1.choices[0].message.content)

=== system 없음 (범용 설명, 길어질 수 있음) ===
`for`문은 프로그래밍에서 반복문 중 하나로, 특정한 작업을 여러 번 반복할 때 사용됩니다. 주로 리스트, 배열, 또는 범위와 같은 반복 가능한 객체의 요소를 순회하면서 작업을 수행할 때 사용됩니다.

예를 들어, Python에서 `for`문을 사용하는 기본적인 예시는 다음과 같습니다:

```python
# 0부터 4까지의 숫자를 출력
for i in range(5):
    print(i)
```

위의 코드에서 `range(5)`는 0부터 4까지의 숫자를 생성하고, `for`문은 이 숫자들을 하나씩 `i`에 할당하여 반복적으로 `print(i)`를 실행합니다.

`for`문은 다양한 프로그래밍 언어에서 비슷한 방식으로 사용되며, 각 언어마다 문법이 조금씩 다를 수 있습니다.


In [36]:
r2 = client.chat.completions.create(
    model='gpt-4o-mini',
    temperature=0.2,
    messages=[
        {
            'role': 'system',
            'content': '''You are a Python tutor.
Rules:
- Answer in Korean only
- Use exactly 3 lines: [비유 1문장] / [코드 예시 max 3 lines] / [실무 한 줄]
- No long explanation, no bullet lists''',
        },
        {'role': 'user', 'content': question},
    ],
)

print('=== system 있음 (튜터 + 3줄 형식 강제) ===')
print(r2.choices[0].message.content)

=== system 있음 (튜터 + 3줄 형식 강제) ===
for문은 반복하는 기계처럼, 주어진 작업을 여러 번 수행하게 해줍니다.  
```python
for i in range(5):
    print(i)
```
반복적인 작업을 간편하게 처리할 수 있습니다.


---
## 6. temperature — **일반 예시**

같은 질문, 다른 `temperature` → 결과가 어떻게 달라지는지 확인합니다.

| temperature | 특징 | 용도 |
|-------------|------|------|
| 0 ~ 0.3 | 일관적, 사실 위주 | 분석, 판정, 코드 |
| 0.7 ~ 1.0 | 다양·창의적 | 브레인스토밍, 카피 |

In [37]:
question = '팀 워크숍 아이스브레이킹 아이디어 1가지만.'

for temp in [0.0, 0.7, 1.0]:
    r = client.chat.completions.create(
        model='gpt-4o-mini',
        temperature=temp,
        messages=[{'role': 'user', 'content': question}],
    )
    print(f'[temperature={temp}] {r.choices[0].message.content}')
    print()

[temperature=0.0] "두 진실과 한 거짓" 게임을 제안합니다. 각 팀원이 자신에 대한 두 가지 진실과 하나의 거짓을 말합니다. 다른 팀원들은 어떤 것이 거짓인지 맞춰보는 방식입니다. 이 게임은 서로에 대한 이해를 높이고, 자연스럽게 대화를 유도하는 데 도움이 됩니다. 재미있고 가벼운 분위기를 만들어 팀원 간의 친밀감을 증진시킬 수 있습니다.

[temperature=0.7] "두 진실과 한 거짓" 게임을 제안합니다. 각 팀원이 자신에 대한 두 가지 진실과 하나의 거짓을 준비합니다. 차례로 자신이 만든 세 가지 이야기를 공유하고, 다른 팀원들은 어떤 것이 거짓인지 맞추는 게임입니다. 이 활동은 서로에 대한 이해를 높이고, 팀원 간의 유대감을 강화하는 데 도움이 됩니다.

[temperature=1.0] "두 진실과 한 거짓" 게임을 제안합니다. 각 팀원이 자신에 대한 세 가지 사실을 이야기하는데, 그 중 두 가지는 진실이고 하나는 거짓입니다. 다른 팀원들은 어떤 것이 거짓인지 추측합니다. 이 활동은 서로에 대한 이해를 높이고, 팀원 간의 대화를 촉진하는 데 도움이 됩니다. 재미있고 가벼운 분위기에서 자연스럽게 친밀감을 형성할 수 있습니다!



In [38]:
question = '대한민국 최고의 운동 선수는 누구일까'

for temp in [0.0, 0.7, 1.0]:
    r1 = client.chat.completions.create(
        model='gpt-4o-mini',
        temperature=temp,
        messages=[{'role': 'user', 'content': question}],
    )
    print(f'[temperature={temp}] {r1.choices[0].message.content}')
    print()

[temperature=0.0] 대한민국 최고의 운동선수는 여러 분야에서 다양한 인물들이 있을 수 있습니다. 예를 들어, 축구에서는 손흥민이 세계적으로 유명하며, 야구에서는 박찬호와 류현진이 많은 사랑을 받고 있습니다. 또한, 올림픽에서 금메달을 딴 김연아와 같은 피겨 스케이팅 선수도 많은 존경을 받고 있습니다. 각 스포츠마다 최고의 선수는 다를 수 있으니, 어떤 종목을 기준으로 하느냐에 따라 달라질 수 있습니다. 당신이 생각하는 최고의 운동선수는 누구인가요?

[temperature=0.7] 대한민국 최고의 운동선수를 선정하는 것은 주관적인 의견이 많이 반영되는 문제입니다. 여러 분야에서 뛰어난 성과를 낸 선수들이 많기 때문입니다. 예를 들어:

1. **축구**: 손흥민은 현재 대한민국에서 가장 유명한 축구 선수 중 한 명으로, 프리미어 리그 토트넘 홋스퍼에서 뛰고 있으며 많은 팬들에게 사랑받고 있습니다.
2. **야구**: 박찬호는 MLB에서 활동하며 대한민국 최초의 메이저리그 투수로서 큰 성과를 이뤘습니다.
3. **배드민턴**: 우즈베키스탄에서 열린 2019 세계 배드민턴 선수권 대회에서 금메달을 획득한 여자 배드민턴 선수인 이재경도 주목받고 있습니다.
4. **올림픽**: 양궁, 유도, 체조 등 다양한 종목에서 올림픽 금메달을 딴 선수들도 많습니다.

각 분야별로 뛰어난 선수들이 있으며, 개인의 취향이나 기준에 따라 최고의 선수는 달라질 수 있습니다.

[temperature=1.0] 대한민국 최고의 운동 선수는 여러 분야와 시대에 따라 다르게 평가될 수 있지만, 몇몇 선수들이 특히 두각을 나타내고 있습니다. 

1. **박지성**: 축구 선수로, 세계적인 클럽인 맨체스터 유나이티드에서 활약하며 많은 기록을 세웠습니다.
2. **김연아**: 피겨 스케이팅 선수로, 2010년 올림픽 금메달을 비롯해 여러 차례 세계 선수권 대회에서 우승한 경력이 있습니다.
3. **손흥민**: 최근의 축구 스타로, 토트넘 홋스퍼에서 활약하며 엄청난 인기와 성과를 거두고

---
## 7. `max_tokens` — 출력 길이 제한

답변이 너무 길어지는 것을 막거나, 비용을 줄일 때 사용합니다.

In [39]:
for max_tok in [20, 100]:
    r = client.chat.completions.create(
        model='gpt-4o-mini',
        temperature=0.3,
        max_tokens=max_tok,
        messages=[{'role': 'user', 'content': '대한민국의 역사를 요약해줘.'}],
    )
    print(f'--- max_tokens={max_tok} (finish: {r.choices[0].finish_reason}) ---')
    print(r.choices[0].message.content)
    print()

--- max_tokens=20 (finish: length) ---
대한민국의 역사는 고대부터 현대에 이르기까지 다양한 사건과 변화를 �

--- max_tokens=100 (finish: length) ---
대한민국의 역사는 고대부터 현대에 이르기까지 다양한 사건과 변화를 겪어왔습니다. 아래는 주요 역사적 사건들을 요약한 것입니다.

1. **고대 및 삼국시대 (기원전 2333년 ~ 668년)**: 
   - 고조선이 기원전 2333년에 건국되었다고 전해지며, 이후 삼국(고구려, 백제, 신라)이 형



---
## 8. `ask()` 함수 — 재사용 패턴

같은 API 호출을 함수로 감싸기.

In [40]:
def ask(user_message, system_message='You are a helpful assistant.', temperature=0.2, max_tokens=None):
    """1턴 질의 — 답변 텍스트만 반환"""
    kwargs = dict(
        model='gpt-4o-mini',
        temperature=temperature,
        messages=[
            {'role': 'system', 'content': system_message},
            {'role': 'user',   'content': user_message},
        ],
    )
    if max_tokens is not None:
        kwargs['max_tokens'] = max_tokens
    response = client.chat.completions.create(**kwargs)
    return response.choices[0].message.content


print(ask('서울 3일 여행 코스 추천해줘. bullet 3개만.', max_tokens=150))

서울 3일 여행 코스 추천드립니다:

- **1일차: 역사와 문화 탐방**
  - 경복궁 방문 및 국립민속박물관 관람
  - 북촌 한옥마을 산책
  - 인사동 거리에서 전통 차와 기념품 쇼핑

- **2일차: 현대 서울 경험하기**
  - 동대문 디자인 플라자(DDP) 탐방
  - 명동 쇼핑 및 길거리 음식 체험
  - 남산타워(N서울타워)에서 서울 야경 감상

- **3일차: 자연과 여유 즐기기**
  - 한강공원에서 자전거 타


In [42]:
print(ask('내 이름은 이창주야', max_tokens=150))

안녕하세요, 이창주님! 어떻게 도와드릴까요?


In [43]:
print(ask('내이름이 뭐라고?', max_tokens=150))

죄송하지만, 당신의 이름을 알 수 있는 정보가 없습니다. 당신의 이름을 알려주시면 그에 맞춰 대화할 수 있습니다!


---
## 9. 멀티턴 대화 

이전 대화를 `messages`에 **그대로 이어 붙이면** 후속 질문이 가능합니다.

```
system → user → assistant → user → ...
```

### 기존대화

In [41]:
messages = [
    {'role': 'system', 'content': 'You are a helpful travel assistant. Answer in Korean.'},
    {'role': 'user',   'content': '제주도 2박 3일 여행 계획 짜줘. 하루 요약 1줄씩.'},
]

r1 = client.chat.completions.create(model='gpt-4o-mini', temperature=0.3, messages=messages)
answer1 = r1.choices[0].message.content
print('=== 1턴 ===')
print(answer1)

messages = [
    {'role': 'system', 'content': 'You are a helpful travel assistant. Answer in Korean.'},
    {'role': 'user',   'content': '2일차만 더 자세히. 맛집 2곳 포함.'},
]

r2 = client.chat.completions.create(model='gpt-4o-mini', temperature=0.3, messages=messages)
print('\n=== 2턴 (맥락 유지) ===')
print(r2.choices[0].message.content)

=== 1턴 ===
물론입니다! 제주도 2박 3일 여행 계획을 아래와 같이 짜보았습니다.

### 1일차: 
제주공항 도착 후, 한라산 국립공원에서 하이킹을 즐기고, 저녁에는 제주 전통 음식인 흑돼지 바비큐를 맛본다.

### 2일차: 
성산일출봉에서 일출을 감상한 후, 우도 섬으로 가서 자전거 투어를 즐기고, 저녁에는 해변 근처의 신선한 해산물로 저녁을 해결한다.

### 3일차: 
제주 민속촌에서 제주 전통 문화를 체험하고, 마지막으로 협재 해수욕장에서 여유롭게 바다를 즐긴 후 공항으로 이동한다. 

즐거운 여행 되세요!

=== 2턴 (맥락 유지) ===
2일차 일정은 다음과 같이 구성해 보았습니다.

### 2일차: 서울 탐방

**오전: 경복궁 방문**
- **시간**: 9:00 AM - 11:00 AM
- **내용**: 경복궁은 조선 왕조의 주요 궁궐로, 아름다운 건축물과 정원이 인상적입니다. 경복궁의 근정전과 경회루를 둘러보며 한국의 전통 건축을 감상하세요. 또한, 오전 10시에 열리는 수문장 교대식도 놓치지 마세요.

**점심: 맛집 1 - 광화문 '한일관'**
- **시간**: 11:30 AM - 12:30 PM
- **추천 메뉴**: 불고기 정식 또는 비빔밥
- **설명**: 전통 한식을 제공하는 식당으로, 신선한 재료와 정갈한 맛이 특징입니다. 경복궁에서 가까워 이동이 편리합니다.

**오후: 북촌 한옥마을 탐방**
- **시간**: 1:00 PM - 3:00 PM
- **내용**: 전통 한옥이 잘 보존된 북촌 한옥마을을 거닐며 한국의 전통 문화를 느껴보세요. 다양한 공방과 카페도 있어 기념품을 구매하거나 휴식을 취하기 좋습니다.

**오후 간식: 맛집 2 - '북촌손만두'**
- **시간**: 3:30 PM - 4:00 PM
- **추천 메뉴**: 손만두, 만두국
- **설명**: 북촌 한옥마을 근처에 위치한 만두 전문점으로, 수제 만두가 인기입니다. 따뜻한 만두로 오후 간식을 즐겨보세요.

**저녁: 명동 쇼핑 및 저녁식사**
- **시

### 멀티턴

### Assistant 메시지는 앞서 있었던 prompt들(user/system)에 대한 모델의 응답을 구성하며, 종종 대화 상태를 유지하기 위해 히스토리의 일부로 저장됩니다.

#### 특징
모델이 생성한 응답을 포함한다.

이전 message의 context 처리를 위해 대화의 연속성 유지를 돕는다.

다음 API 호출에서는 assistant 메시지를 포함시켜야 문맥이 이어짐.

In [56]:
messages = [
    {'role': 'system', 'content': 'You are a helpful travel assistant. Answer in Korean.'},
    {'role': 'user',   'content': '제주도 2박 3일 여행 계획 짜줘. 하루 요약 1줄씩.'},
]

r1 = client.chat.completions.create(model='gpt-4o-mini', temperature=0.3, messages=messages)
answer1 = r1.choices[0].message.content
print('=== 1턴 ===')
print(answer1)
#===========================================================================
#answer1 답변을 assistant에 넣어야 함
#messages = [
#    {'role': 'system', 'content': 'You are a helpful travel assistant. Answer in Korean.'},
#    {'role': 'assistant', 'content': answer1},
#    {'role': 'user',   'content': '제주도 2박 3일 여행 계획 짜줘. 하루 요약 1줄씩.'},
#]

messages.append({'role': 'assistant', 'content': answer1})
messages.append({'role': 'user', 'content': '2일차만 더 자세히. 맛집 2곳 포함.'})

r2 = client.chat.completions.create(model='gpt-4o-mini', temperature=0.3, messages=messages)
print('\n=== 2턴 (맥락 유지) ===')
print(r2.choices[0].message.content)

=== 1턴 ===
제주도 2박 3일 여행 계획은 다음과 같습니다:

**1일차:** 제주공항 도착 후 한라산 등반 및 성산일출봉 관람, 저녁에는 서귀포시에서 해산물 저녁식사.

**2일차:** 우도 섬으로 페리 이동 후 자전거 투어, 해변에서 수영 및 우도 땅콩 아이스크림 맛보기, 저녁에는 제주 흑돼지 바비큐.

**3일차:** 제주 민속촌 방문 후 제주도 전통 시장 탐방, 공항으로 이동하여 기념품 쇼핑 후 출발.

=== 2턴 (맥락 유지) ===
**2일차 상세 계획:**

- **아침:** 제주도에서 아침을 먹고 우도 섬으로 페리 이동. 
  - 추천 맛집: **우도 땅콩 아이스크림** - 우도의 유명한 땅콩 아이스크림을 꼭 맛보세요.

- **오전:** 우도에서 자전거를 대여하여 섬을 둘러보며 아름다운 해변과 경치를 감상합니다. 
  - 추천 코스: 검멀레 해변, 하우목동 해변.

- **점심:** 우도에서 현지 식사.
  - 추천 맛집: **우도 뚜레쥬르** - 신선한 해산물로 만든 뚝배기 갈치조림이나 해물 파전을 추천합니다.

- **오후:** 해변에서 수영이나 스노클링을 즐기고, 우도의 다양한 명소를 탐방합니다. 
  - 추천 명소: 우도 봉수대, 서빈백사.

- **저녁:** 제주 본섬으로 돌아와 제주 흑돼지 바비큐를 즐깁니다.
  - 추천 맛집: **돈사돈** - 신선한 제주 흑돼지를 맛볼 수 있는 인기 있는 식당입니다.

- **저녁 후:** 서귀포 시내를 산책하며 제주도의 야경을 감상합니다. 

이렇게 2일차를 알차게 보내시면 좋겠습니다!


---
## ✏️ 실습 1

아래 주제로 **2턴** 대화를 만들어 보세요.

1턴: "Python for문 기본 문법 설명해줘"
2턴: "방금 설명한 걸로 1~5 합 구하는 코드 예시만 보여줘"

In [51]:
!pip install streamlit

In [58]:
# ── 여기에 작성 ──
messages = [
    {'role': 'system', 'content': 'You are a Python tutor. Answer in Korean.'},
    {'role': 'user', 'content': 'Python for문 기본 문법 설명해줘'},
]
# r1 = client.chat.completions.create(...)
r1 = client.chat.completions.create(model='gpt-4o-mini', temperature=0.3, messages=messages)
#print(r1.choices[0].message.content)
answer1 = r1.choices[0].message.content #r1에 대한 질문 답변

messages.append({'role': 'assistant', 'content': answer1}) #맥락 유지
messages.append({'role': 'user', 'content': '방금 설명한 걸로 1~5 합 구하는 코드 예시만 보여줘'}) #새로운 질문

r2 = client.chat.completions.create(model='gpt-4o-mini', temperature=0.3, messages=messages)
print(r2.choices[0].message.content)

1부터 5까지의 합을 구하는 코드는 다음과 같이 작성할 수 있습니다:

```python
합계 = 0
for i in range(1, 6):  # 1부터 5까지의 숫자를 생성
    합계 += i  # 합계에 i를 더함

print(합계)  # 결과 출력
```

이 코드를 실행하면 1부터 5까지의 합인 15가 출력됩니다.


---
## ✏️ 실습 1

커서 프롬프트 : 3일차 폴더에 스트림릿과 실습코드의 openai api를 이용해서 간단한 챗봇을 만드는 python 코드 만들어줘

In [50]:
!pip install streamlit